# Synthetic Data for Testing

In [1]:
from watttime.evaluator.sessions import SessionsGenerator

## Example: At home EV charging

- Covers a 5.5 - 8.5 hour variable length window
- The vehicle has a BMW and has an average power draw of 42.5
- Battery is usually typically 50% charged at plug in time.
- Charging occurs during the workday

In [2]:
ev_kwargs = {
    "max_power_output_rates": [42.5],
    "max_percent_capacity": 0.95,  # highest level of charge achieved by battery
    "power_output_efficiency": 1,  # power loss. 1 = no power loss.
    "minimum_battery_starting_capacity": 0.2,  # minimum starting percent charged
    "minimum_usage_window_start_time": "08:00:00",  # session can start as early as 8am
    "maximum_usage_window_start_time": "22:00:00",  # session can start as late as 9pm
}

In [3]:
s_ev = SessionsGenerator(**ev_kwargs)

Now we can generate synthetic data for users and devices with the attributes set above. The synthetic data creates one example per day.

In [4]:
# the class has a helper function to generate a random list of unique dates
distinct_date_list = s_ev.assign_random_dates(years=[2025])

You can generate data from a single user.

In [7]:
s_ev.synthetic_user_data(distinct_date_list=[distinct_date_list[0]]).T

,0
distinct_dates,2025-01-06
user_type,r27.7525_tc55_avglc25812_sdlc7930
usage_window_start,2025-01-06 19:15:00
usage_window_end,2025-01-07 03:30:00
initial_charge,0.607095
usage_time_required_in_minutes,40.774277
expected_baseline_charge_complete_timestamp,2025-01-06 19:55:46.456627530
window_length_in_minutes,495.0
final_charge_time,2025-01-06 19:55:46.456627530
total_capacity,55


In [8]:
distinct_date_list[0]

datetime.date(2025, 1, 6)

Or for multiple users.

In [ ]:
s_ev.generate_synthetic_dataset(distinct_date_list=[distinct_date_list[0]], number_of_users=10)

100%|██████████| 1000/1000 [00:02<00:00, 460.70it/s]


## Example: AI Model Training
- Model training can occur at any time of day
- There are 3 server models that consume 240, 310, and 640 watt-hour on average
- Early stopping is not an option

In [9]:
ai_kwargs = {
    "max_percent_capacity":1.0, # job must run to completion
    "max_power_output_rates": [24,31,64], # assuming a bare metal usecase, k8s or vm rescale to vCPU
    "minimum_usage_window_start_time": "00:00:00",  # earliest session can start
    "maximum_usage_window_start_time": "23:59:00",  # latest session can start
}

In [10]:
s_ai = SessionsGenerator(**ai_kwargs)

In [11]:
df_ai = s_ai.generate_synthetic_dataset(distinct_date_list=distinct_date_list, number_of_users=10)

100%|██████████| 10/10 [00:00<00:00, 272.29it/s]


# Optimization

In [12]:
import os
import numpy as np
from watttime.evaluator.evaluator import OptChargeEvaluator
from watttime.evaluator.evaluator import ImpactEvaluator
from watttime.evaluator.evaluator import RecalculationOptChargeEvaluator

In [ ]:
username = os.getenv("WATTTIME_USER")
password = os.getenv("WATTTIME_PASSWORD")
region = "CAISO_NORTH"
oce = OptChargeEvaluator(username=username,password=password)
roce = RecalculationOptChargeEvaluator(username,password)

# single instance
df_ev_sample = s_ev.synthetic_user_data(distinct_date_list=[distinct_date_list[0]])
df_ev_sample = df_ev_sample.rename({"usage_time_required_in_minutes":"time_needed"},axis=1)

In [15]:
%%capture
input_dict = df_ev_sample[['usage_window_start',
                    'usage_window_end',
                    'time_needed',
                    'usage_power_kw'
                    ]].T.to_dict()

value = input_dict[0]
value.update({'region':region,'tz_convert':True, "verbose":False})

In [16]:
value

{'usage_window_start': Timestamp('2025-01-06 19:15:00'),
 'usage_window_end': Timestamp('2025-01-07 02:55:00'),
 'time_needed': 51.67684008874637,
 'usage_power_kw': 37.4425,
 'region': 'CAISO_NORTH',
 'tz_convert': True,
 'verbose': False}

In [17]:
df = oce.get_schedule_and_cost_api(**value)
r = ImpactEvaluator(username,password,df).get_all_emissions_values(region=region)

In [18]:
r

{'baseline': 31.979059944544282,
 'forecast': 29.951449198552897,
 'actual': 30.192892487355493}

# Requery

In [19]:
value.update({"optimization_method": "simple", "interval":15, "charge_per_interval":None})

In [20]:
value

{'usage_window_start': Timestamp('2025-01-06 19:15:00'),
 'usage_window_end': Timestamp('2025-01-07 02:55:00'),
 'time_needed': 51.67684008874637,
 'usage_power_kw': 37.4425,
 'region': 'CAISO_NORTH',
 'tz_convert': True,
 'verbose': False,
 'optimization_method': 'simple',
 'interval': 15,
 'charge_per_interval': None}

In [ ]:
%%timeit
df_requery = roce.fit_recalculator(**value)
r_requery = ImpactEvaluator(username,password,df_requery).get_all_emissions_values(region=region)

# Helpful Functions

In [ ]:
# 4 seconds per row, mostly API call
def analysis_loop(region, input_dict):
    results = {}
    for key,value in input_dict.items():
        value.update({'region':region,'tz_convert':True, "verbose":False})
        df = oce.get_schedule_and_cost_api(**value)
        m, b = np.polyfit(np.arange(len(df.pred_moer.values)),df.pred_moer.values, 1)
        stddev = df.pred_moer.std()
        r = ImpactEvaluator(username,password,df).get_all_emissions_values(region=region)
        r.update({'m':m,'b':b,'stddev':stddev})
        results.update({key:r})
    return results

# 4 seconds per row, mostly API call
def analysis_loop_requery(region, input_dict, interval):
    results = {}
    for key,value in tqdm.tqdm(input_dict.items()):
        value.update(
            {'region':region,
            'tz_convert':True, 
            "optimization_method": "simple", 
            "verbose":False,
            "interval":interval,
            "charge_per_interval":None}
            )
        df = roce.fit_recalculator(**value)
        m, b = np.polyfit(np.arange(len(df.pred_moer.values)),df.pred_moer.values, 1)
        stddev = df.pred_moer.std()
        r = ImpactEvaluator(username,password,df).get_all_emissions_values(region=region)
        r.update({'m':m,'b':b,'stddev':stddev})
        results.update({key:r})
    return results

In [ ]:
df_many_sessions = s_ev.generate_synthetic_dataset(distinct_date_list=[distinct_date_list[0]], number_of_users=10)

input_dict = df_many_sessions[['usage_window_start',
                               'usage_window_end',
                                'time_needed',
                                'usage_power_kw'
                                ]].T.to_dict()

# no requery
df_many_sessions_optimized = analysis_loop(region, input_dict)

# requery
df_many_sessions_optimized_requery = analysis_loop_requery(region, input_dict, 45)

# pd.concat([df_many_sessions, df_many_sessions_optimized]) or pd.concat([df_many_sessions, df_many_sessions_optimized_requery])